In [0]:
# ============================================================
# Silver mappings: how ONE bronze table fills ONE silver entity (proposed by the Data Quality Agent).
# One YAML file per (data source, entity): silver_specs/<source_name>__<entity>.yml
# Every contract column must be declared: {from: ...} or {constant: ...} ('constant: null' = no data).
# Transformations are a closed list (no free code): timezone, format, map_values, multiply, constant.
# This notebook loads and validates mappings statically (no Spark), against the entity contract
# and the bronze source spec. Execution on real data comes in the silver framework.
# Requires: common/_utils_config, common/_utils_reference_data, silver/utils/_utils_silver_contracts
# Cells: 1 constants/load · 2 validation · 3 description for review
# ============================================================

silver_specs_folder = "silver_specs"
mapping_name_separator = "__"
root_alias = "payload"                       # the bronze VARIANT column
location_code_field = "location_code"        # built-in: bronze location_codes[position]
timezone_modes = {"offset", "UTC"}
iana_timezone_pattern = r"[A-Z][A-Za-z_]+(/[A-Za-z0-9_\-+]+){1,2}"
alias_pattern = r"[a-z][a-z0-9_]{0,40}"
path_segment_pattern = r"[A-Za-z0-9_\-]+"
csv_column_pattern = r"[a-z][a-z0-9_]{0,60}"

allowed_mapping_keys = {"mapping", "entity", "source", "parse", "columns", "source_record_id"}
allowed_mapping_source_keys = {"source_name", "endpoint"}
allowed_column_rule_keys = {"from", "constant", "timezone", "format", "map_values", "multiply"}
from_only_rule_keys = {"timezone", "format", "map_values", "multiply"}


def load_mapping(yaml_text):
    try:
        mapping = yaml.safe_load(yaml_text)
    except yaml.YAMLError as e:
        raise ValueError(f"Invalid YAML: {str(e)[:300]}")
    if not isinstance(mapping, dict):
        raise ValueError("YAML mapping must be a mapping (key: value) at the top level")
    return mapping


def list_mappings():
    folder = find_repo_folder(silver_specs_folder)
    return sorted(f[:-4] for f in os.listdir(folder) if f.endswith(".yml"))


def load_mapping_file(mapping_name):
    path = os.path.join(find_repo_folder(silver_specs_folder), f"{mapping_name}.yml")
    with open(path, encoding="utf-8") as f:
        mapping = load_mapping(f.read())
    if mapping.get("mapping") != mapping_name:
        raise ValueError(f"{path}: mapping '{mapping.get('mapping')}' must match the file name")
    return mapping


def load_all_mappings():
    """All mappings by name."""
    return {name: load_mapping_file(name) for name in list_mappings()}


def find_bronze_endpoint(source_name, endpoint_name):
    """Returns (bronze source spec, endpoint spec); None where not found."""
    for spec in list_bronze_source_specs():
        if isinstance(spec, dict) and spec.get("source_name") == source_name:
            for endpoint in spec.get("endpoints") or []:
                if isinstance(endpoint, dict) and endpoint.get("name") == endpoint_name:
                    return spec, endpoint
            return spec, None
    return None, None


def bronze_table_for_mapping(mapping):
    source = mapping["source"]
    return f"{catalog}.{bronze_schema}.{source['source_name']}_{source['endpoint']}"

In [0]:
# ------------------------------------------------------------
# Static validation of a mapping (pure logic, no Spark)
# aliases: {alias: None (any field below it) | set of fields (zip_explode: one field per array)}
# ------------------------------------------------------------
def split_path(path):
    """'alias.a.b' -> ('alias', ['a', 'b']); (None, None) if malformed."""
    if not isinstance(path, str):
        return None, None
    parts = path.split(".")
    if not re.fullmatch(alias_pattern, parts[0]) or not all(re.fullmatch(path_segment_pattern, p) for p in parts[1:]):
        return None, None
    return parts[0], parts[1:]


def validate_path(path, aliases, label):
    alias, fields = split_path(path)
    if alias is None:
        return [f"{label}: '{path}' is not a valid path (alias.field.subfield)"]
    if alias not in aliases:
        return [f"{label}: unknown alias '{alias}' in '{path}' (defined: {sorted(aliases)})"]
    allowed_fields = aliases[alias]
    if allowed_fields is not None and (len(fields) != 1 or fields[0] not in allowed_fields):
        return [f"{label}: '{path}' must be {alias}.<field> with a field that is one of the zip_explode arrays "
                f"{sorted(allowed_fields)}"]
    return []


def validate_mapping_source(mapping):
    """Returns (errors, bronze endpoint spec or None)."""
    source = mapping.get("source")
    if not isinstance(source, dict):
        return ["source must be a mapping with source_name and endpoint"], None
    errors = []
    unknown = set(source) - allowed_mapping_source_keys
    if unknown:
        errors.append(f"source: unknown keys {sorted(unknown)}")
    source_name, endpoint_name = source.get("source_name"), source.get("endpoint")
    spec, endpoint = find_bronze_endpoint(source_name, endpoint_name)
    if spec is None:
        return errors + [f"source: '{source_name}' has no bronze source spec in {source_specs_folder}/"], None
    if endpoint is None:
        names = [ep.get("name") for ep in spec.get("endpoints") or [] if isinstance(ep, dict)]
        return errors + [f"source: '{source_name}' has no endpoint '{endpoint_name}' (endpoints: {names})"], None
    return errors, endpoint


def validate_parse_steps(parse, endpoint):
    """Returns (errors, aliases, has_position)."""
    is_csv = endpoint.get("response_format") == "csv"
    if parse in (None, []):
        return [], ({} if is_csv else {root_alias: None}), False
    if is_csv:
        return ["parse only applies to json endpoints (csv rows are already columns)"], {}, False
    if not isinstance(parse, list):
        return ["parse must be a list of steps"], {root_alias: None}, False

    errors, aliases, has_position = [], {root_alias: None}, False
    for i, step in enumerate(parse):
        label = f"parse step #{i + 1}"
        if not isinstance(step, dict) or len({"explode", "zip_explode"} & set(step)) != 1:
            errors.append(f"{label}: must be either an explode or a zip_explode step")
            continue

        if "explode" in step:
            new_alias, fields = step.get("as"), None
            errors += validate_path(step["explode"], aliases, label)
            unknown = set(step) - {"explode", "as", "position"}
            if "position" in step:
                if not endpoint.get("locations"):
                    errors.append(f"{label}: position is only allowed when the bronze endpoint has a locations block")
                elif has_position:
                    errors.append(f"{label}: only one step can declare position")
                elif not isinstance(step["position"], str) or not re.fullmatch(alias_pattern, step["position"]):
                    errors.append(f"{label}: position must be a snake_case name")
                else:
                    has_position = True
        else:
            body = step["zip_explode"]
            if not isinstance(body, dict):
                errors.append(f"{label}: zip_explode must be a mapping with as and arrays")
                continue
            new_alias, arrays = body.get("as"), body.get("arrays")
            unknown = (set(step) - {"zip_explode"}) | (set(body) - {"as", "arrays"})
            if not isinstance(arrays, dict) or not arrays:
                errors.append(f"{label}: zip_explode.arrays must be a non-empty mapping <field>: <path>")
                fields = set()
            else:
                for field, path in arrays.items():
                    if not re.fullmatch(alias_pattern, str(field)):
                        errors.append(f"{label}: array field '{field}' must be snake_case")
                    errors += validate_path(path, aliases, f"{label} array {field}")
                fields = {str(field) for field in arrays}

        if unknown:
            errors.append(f"{label}: unknown keys {sorted(unknown)}")
        if not isinstance(new_alias, str) or not re.fullmatch(alias_pattern, new_alias):
            errors.append(f"{label}: 'as' must be a snake_case alias")
        elif new_alias in aliases or new_alias == location_code_field:
            errors.append(f"{label}: duplicated alias '{new_alias}'")
        else:
            aliases[new_alias] = fields
    return errors, aliases, has_position


def validate_source_expression(value, aliases, has_position, is_csv, label):
    """A 'from' value: location_code, a csv column (csv endpoints) or an alias path (json endpoints)."""
    if value == location_code_field:
        if has_position:
            return []
        return [f"{label}: '{location_code_field}' requires a parse step with position "
                f"(bronze endpoint with a locations block)"]
    if is_csv:
        if isinstance(value, str) and re.fullmatch(csv_column_pattern, value):
            return []
        return [f"{label}: '{value}' must be a bronze csv column name (snake_case)"]
    return validate_path(value, aliases, label)


def validate_map_values(label, values, column):
    """map_values: source codes (strings) -> canonical values (strings for string columns, numbers for numeric)."""
    column_type = column.get("type")
    numeric_target = is_numeric_type(column_type)
    if column_type != "string" and not numeric_target:
        return [f"{label}: map_values only applies to string or numeric columns"]
    if not isinstance(values, dict) or not values or not all(isinstance(k, str) for k in values):
        return [f"{label}: map_values must map source codes to canonical values; source codes are strings "
                f"(quote numeric codes)"]
    if numeric_target:
        if not all(is_number(v) for v in values.values()):
            return [f"{label}: map_values targets must be numbers for a {column_type} column"]
        value_range = column.get("range") or {}
        outside = sorted(v for v in values.values()
                         if ("min" in value_range and v < value_range["min"])
                         or ("max" in value_range and v > value_range["max"]))
        if outside:
            return [f"{label}: map_values targets {outside} are outside the range {value_range}"]
        return []
    if not all(isinstance(v, str) for v in values.values()):
        return [f"{label}: map_values targets must be strings for a string column (quote numeric codes)"]
    if column.get("accepted_values"):
        outside = sorted({v for v in values.values() if v not in column["accepted_values"]})
        if outside:
            return [f"{label}: map_values targets {outside} are not in accepted_values {column['accepted_values']}"]
    return []


def validate_column_rule(column_name, rule, column, is_business_key, aliases, has_position, is_csv):
    label = f"column {column_name}"
    if not isinstance(rule, dict):
        return [f"{label}: must be a mapping ({{from: ...}} or {{constant: ...}})"]
    errors = []
    column_type = column.get("type")
    unknown = set(rule) - allowed_column_rule_keys
    if unknown:
        errors.append(f"{label}: unknown keys {sorted(unknown)}")
    if ("from" in rule) == ("constant" in rule):
        return errors + [f"{label}: needs exactly one of from or constant "
                         f"(use 'constant: null' for columns without data in this source)"]

    if "constant" in rule:
        value = rule["constant"]
        transforms = set(rule) & from_only_rule_keys
        if transforms:
            errors.append(f"{label}: {sorted(transforms)} only apply to columns mapped with from")
        if value is None:
            if is_business_key or column.get("nullable") is False:
                errors.append(f"{label}: cannot be null (business key or nullable: false)")
        elif column_type == "string":
            if not isinstance(value, str):
                errors.append(f"{label}: constant must be a string")
            elif column.get("accepted_values") and value not in column["accepted_values"]:
                errors.append(f"{label}: constant '{value}' is not in accepted_values {column['accepted_values']}")
        elif is_numeric_type(column_type):
            if not is_number(value):
                errors.append(f"{label}: constant must be a number")
        elif column_type == "boolean":
            if not isinstance(value, bool):
                errors.append(f"{label}: constant must be true or false")
        else:
            errors.append(f"{label}: constant values are not supported for {column_type} columns (only null)")
        return errors

    errors += validate_source_expression(rule["from"], aliases, has_position, is_csv, label)

    if "timezone" in rule:
        timezone = rule["timezone"]
        if column_type != "timestamp":
            errors.append(f"{label}: timezone only applies to timestamp columns")
        elif not isinstance(timezone, str) or not (timezone in timezone_modes
                                                    or re.fullmatch(iana_timezone_pattern, timezone)):
            errors.append(f"{label}: timezone must be 'offset', 'UTC' or an IANA zone like Europe/Madrid")
    elif column_type == "timestamp":
        errors.append(f"{label}: timestamp columns need timezone ('offset', 'UTC' or an IANA zone) "
                      f"so that they are converted to UTC")

    if "format" in rule:
        if column_type not in ("timestamp", "date"):
            errors.append(f"{label}: format only applies to date and timestamp columns")
        elif not isinstance(rule["format"], str) or not rule["format"].strip():
            errors.append(f"{label}: format must be a datetime pattern like dd/MM/yyyy")

    if "map_values" in rule:
        errors += validate_map_values(label, rule["map_values"], column)

    if "multiply" in rule:
        if not is_numeric_type(column_type):
            errors.append(f"{label}: multiply only applies to numeric columns")
        elif not is_number(rule["multiply"]) or rule["multiply"] == 0:
            errors.append(f"{label}: multiply must be a non-zero number")
    return errors


def validate_mapping(mapping, contracts):
    if not isinstance(mapping, dict):
        return ["mapping must be a mapping (key: value)"]
    errors = []
    unknown = set(mapping) - allowed_mapping_keys
    if unknown:
        errors.append(f"unknown keys {sorted(unknown)}")

    entity = mapping.get("entity")
    contract = contracts.get(entity)
    if contract is None:
        errors.append(f"entity '{entity}' has no contract in {entity_contracts_folder}/")
    elif contract.get("table_type") != "entity":
        errors.append(f"entity '{entity}' is a {contract.get('table_type')} entity; "
                      f"mappings only apply to entities of table_type 'entity'")

    source_errors, endpoint = validate_mapping_source(mapping)
    errors += source_errors
    source_name = mapping["source"].get("source_name") if isinstance(mapping.get("source"), dict) else None
    expected_name = f"{source_name}{mapping_name_separator}{entity}"
    if mapping.get("mapping") != expected_name:
        errors.append(f"mapping must be '{expected_name}' (<source_name>{mapping_name_separator}<entity>)")

    if endpoint is None or contract is None or contract.get("table_type") != "entity":
        return errors                               # columns cannot be checked without source and entity

    parse_errors, aliases, has_position = validate_parse_steps(mapping.get("parse"), endpoint)
    errors += parse_errors
    is_csv = endpoint.get("response_format") == "csv"

    rules = mapping.get("columns")
    if not isinstance(rules, dict) or not rules:
        return errors + ["columns must be a mapping <contract column>: {from: ...} or {constant: ...}"]
    columns = contract_columns(contract)
    missing = [c for c in columns if c not in rules]
    if missing:
        errors.append(f"columns must declare every contract column; missing {missing} "
                      f"(use 'constant: null' for columns without data in this source)")
    extra = [c for c in rules if c not in columns]
    if extra:
        errors.append(f"columns {extra} are not columns of the {entity} contract")

    keys = set(contract.get("business_keys") or [])
    for name, rule in rules.items():
        if name in columns:
            errors += validate_column_rule(name, rule, columns[name], name in keys, aliases, has_position, is_csv)

    if "source_record_id" in mapping:
        errors += validate_source_expression(mapping["source_record_id"], aliases, has_position, is_csv,
                                             "source_record_id")
    return errors


def validate_all_mappings(contracts=None, mappings=None):
    """Returns {mapping: [errors]} only for mappings with errors."""
    contracts = contracts if contracts is not None else load_all_entity_contracts()
    mappings = mappings if mappings is not None else load_all_mappings()
    results = {}
    for name, mapping in mappings.items():
        errors = validate_mapping(mapping, contracts)
        if errors:
            results[name] = errors
    return results

In [0]:
# ------------------------------------------------------------
# Readable lineage of a mapping (bronze -> silver, column by column), for human review
# ------------------------------------------------------------
def describe_mapping(mapping, contracts):
    contract = contracts[mapping["entity"]]
    lines = [f"{bronze_table_for_mapping(mapping)}  ->  {silver_table_full_name(mapping['entity'])}"]
    for step in mapping.get("parse") or []:
        if "explode" in step:
            position = f"  (position {step['position']} -> location_code)" if "position" in step else ""
            lines.append(f"  explode {step['explode']} as {step['as']}{position}")
        else:
            body = step["zip_explode"]
            lines.append(f"  zip_explode as {body['as']}: {', '.join(body['arrays'])}")

    for column in contract["columns"]:
        rule = mapping["columns"].get(column["name"], {})
        if "constant" in rule:
            origin = "null (no data in this source)" if rule["constant"] is None else f"constant {rule['constant']!r}"
        else:
            origin = str(rule.get("from", "?"))
            extras = [f"{key}={rule[key]}" for key in ("timezone", "format", "multiply") if key in rule]
            if "map_values" in rule:
                extras.append(f"map_values {rule['map_values']}")
            if extras:
                origin += "  [" + ", ".join(extras) + "]"
        lines.append(f"  {column['name']:<26} <- {origin}")

    source = mapping["source"]
    lines.append(f"  {'source_name':<26} <- '{source['source_name']}'  (FKDataSource = md5(source_name))")
    lines.append(f"  {'source_endpoint':<26} <- '{source['endpoint']}'")
    if "source_record_id" in mapping:
        lines.append(f"  {'source_record_id':<26} <- {mapping['source_record_id']}")
    return "\n".join(lines)